In [1]:
import sys
sys.path.append("..")

import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import (StratifiedKFold, cross_validate, GridSearchCV,
                                     RandomizedSearchCV, cross_val_predict)
from sklearn.metrics import (precision_score, recall_score, f1_score, fbeta_score,
                             roc_auc_score, confusion_matrix, classification_report)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from src.preprocessing import get_preprocessor

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

X_train, y_train = train.drop(columns=["Churn"]), train["Churn"]
X_test, y_test = test.drop(columns=["Churn"]), test["Churn"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [2]:
candidates = {
    "Plain": Pipeline([
        ("preprocess", get_preprocessor(X_train)),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "Class weights": Pipeline([
        ("preprocess", get_preprocessor(X_train)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "SMOTE": ImbPipeline([
        ("preprocess", get_preprocessor(X_train)),
        ("smote", SMOTE(random_state=42)),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
}

scoring = ["roc_auc", "recall", "precision", "f1"]

rows = {}
for name, pipe in candidates.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows[name] = {m: res[f"test_{m}"].mean() for m in scoring}

pd.DataFrame(rows).T.round(3)

/Users/sonuu/Desktop/telco-churn/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/sonuu/Desktop/telco-churn/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/sonuu/Desktop/telco-churn/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/son

,roc_auc,recall,precision,f1
Plain,0.846,0.543,0.654,0.593
Class weights,0.846,0.803,0.518,0.629
SMOTE,0.845,0.792,0.525,0.631


In [3]:
lr_pipe = Pipeline([
    ("preprocess", get_preprocessor(X_train)),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])

lr_search = GridSearchCV(
    lr_pipe,
    param_grid={"model__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
    scoring="roc_auc", cv=cv, n_jobs=-1,
)
lr_search.fit(X_train, y_train)

print("Best params:", lr_search.best_params_)
print("Best CV ROC-AUC:", round(lr_search.best_score_, 4))

Best params: {'model__C': 10}
Best CV ROC-AUC: 0.8462


In [ ]:
gb_pipe = Pipeline([
    ("preprocess", get_preprocessor(X_train)),
    ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=42)),
])

gb_params = {
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__max_iter": [100, 200, 300],
    "model__min_samples_leaf": [20, 50, 100],
    "model__l2_regularization": [0, 1, 5, 10],
}

gb_search = RandomizedSearchCV(
    gb_pipe, gb_params, n_iter=20,
    scoring="roc_auc", cv=cv, n_jobs=-1, random_state=42,
)
gb_search.fit(X_train, y_train)

print("Best params:", gb_search.best_params_)
print("Best CV ROC-AUC:", round(gb_search.best_score_, 4))